# Quand la taille de l'échantillon augmente

Nous simulons des suites de 5 000 valeurs gaussiennes avec :

- moyenne théorique `mu = 100` ;
- écart-type théorique `sigma = 15`.

Pour chaque taille `n`, nous gardons les `n` premières valeurs du
même tirage. Ces échantillons sont donc emboîtés. Nous affichons
20 valeurs de `n`, soit deux fois plus que dans la première version.

**Avant d'exécuter :** les courbes vont-elles se rapprocher de leur
ligne théorique à chaque étape ? Vont-elles finir exactement sur
cette ligne ? Que change le `seed` ?

## 1. Préparer les tailles et une fonction de calcul

`sample_mean` est la moyenne observée. `sample_std` est l'écart-type
observé, calculé avec le dénominateur `n - 1` (`ddof=1`).

Pré-requis : NumPy, pandas et Plotly.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

population_mean = 100.0
population_std = 15.0
sample_sizes = np.array(
    [
        10, 15, 20, 30, 50, 75, 100, 150, 200, 300,
        500, 750, 1_000, 1_500, 2_000, 2_500, 3_000, 3_500, 4_000, 5_000,
    ]
)
major_sample_sizes = np.array([10, 20, 50, 100, 200, 500, 1_000, 2_000, 5_000])
colors = ["#002D62", "#D94801", "#007F73", "#7B2CBF", "#8A6500"]


def convergence_summary(seed):
    # Un seul tirage par seed ; chaque taille reprend ses n premières valeurs.
    rng = np.random.default_rng(seed)
    sample_values = rng.normal(population_mean, population_std, sample_sizes[-1])
    observed_means = []
    observed_stds = []
    for sample_size in sample_sizes:
        current_sample = sample_values[:sample_size]
        observed_means.append(current_sample.mean())
        observed_stds.append(current_sample.std(ddof=1))
    return pd.DataFrame(
        {
            "seed": seed,
            "sample_size": sample_sizes,
            "sample_mean": observed_means,
            "sample_std": observed_stds,
        }
    )


def style_figure(figure, title, y_title):
    figure.update_layout(
        template="plotly_white",
        width=1_200,
        height=560,
        title=title,
        xaxis_title="Taille n de l'échantillon — échelle logarithmique",
        yaxis_title=y_title,
        legend={"orientation": "h", "y": 1.02, "x": 1, "xanchor": "right"},
    )
    figure.update_xaxes(
        type="log",
        tickvals=major_sample_sizes,
        ticktext=[f"{value:,}".replace(",", " ") for value in major_sample_sizes],
    )
    return figure

## 2. Moyenne observée avec `seed = 406`

L'axe horizontal est logarithmique pour rendre visibles les petites
et les grandes tailles sur le même graphique.

In [ ]:
summary_406 = convergence_summary(seed=406)
mean_406 = go.Figure()
mean_406.add_scatter(
    x=summary_406["sample_size"],
    y=summary_406["sample_mean"],
    mode="lines+markers",
    name="Moyenne observée · seed = 406",
    line={"color": colors[0], "width": 3},
)
mean_406.add_scatter(
    x=[sample_sizes.min(), sample_sizes.max()],
    y=[population_mean, population_mean],
    mode="lines",
    name="mu théorique = 100",
    line={"color": "#202A33", "width": 3, "dash": "dash"},
)
style_figure(mean_406, "seed = 406 · 20 tailles de n", "Moyenne observée")
mean_406.show()

**À observer :** la moyenne fluctue et peut temporairement
s'éloigner de 100. Quand `n` augmente, elle devient généralement
plus stable autour de 100 ; le rapprochement n'est pas monotone.

## 3. Écart-type observé avec le même `seed = 406`

In [ ]:
std_406 = go.Figure()
std_406.add_scatter(
    x=summary_406["sample_size"],
    y=summary_406["sample_std"],
    mode="lines+markers",
    name="Écart-type observé s · seed = 406",
    line={"color": colors[0], "width": 3},
)
std_406.add_scatter(
    x=[sample_sizes.min(), sample_sizes.max()],
    y=[population_std, population_std],
    mode="lines",
    name="sigma théorique = 15",
    line={"color": "#202A33", "width": 3, "dash": "dash"},
)
style_figure(std_406, "seed = 406 · 20 tailles de n", "Écart-type observé s")
std_406.show()

**À observer :** l'écart-type peut rester au-dessus ou au-dessous
de 15. Une grande taille le rend généralement plus stable, pas
exactement égal à `sigma`.

## 4. Recommencer avec `seed = 407`

Le modèle de population ne change pas. Seule la suite pseudo-aléatoire
change ; la trajectoire observée doit donc être différente.

In [ ]:
summary_407 = convergence_summary(seed=407)
mean_407 = go.Figure()
mean_407.add_scatter(
    x=summary_407["sample_size"],
    y=summary_407["sample_mean"],
    mode="lines+markers",
    name="Moyenne observée · seed = 407",
    line={"color": colors[1], "width": 3},
)
mean_407.add_scatter(
    x=[sample_sizes.min(), sample_sizes.max()],
    y=[population_mean, population_mean],
    mode="lines",
    name="mu théorique = 100",
    line={"color": "#202A33", "width": 3, "dash": "dash"},
)
style_figure(mean_407, "seed = 407 · 20 tailles de n", "Moyenne observée")
mean_407.show()

## 5. Superposer cinq trajectoires

Chaque couleur correspond à un `seed` différent. La ligne en
tirets reste la même moyenne théorique `mu = 100`.

In [ ]:
seeds = [406, 407, 408, 409, 410]
summaries = {seed: convergence_summary(seed) for seed in seeds}
multiple_means = go.Figure()
for seed, color in zip(seeds, colors, strict=True):
    summary = summaries[seed]
    multiple_means.add_scatter(
        x=summary["sample_size"],
        y=summary["sample_mean"],
        mode="lines+markers",
        name=f"seed = {seed}",
        showlegend=False,
        line={"color": color, "width": 2.5},
    )
multiple_means.add_scatter(
    x=[sample_sizes.min(), sample_sizes.max()],
    y=[population_mean, population_mean],
    mode="lines",
    name="mu théorique = 100",
    line={"color": "#202A33", "width": 3, "dash": "dash"},
)
style_figure(
    multiple_means,
    "5 seeds · 20 tailles de n par trajectoire",
    "Moyenne observée",
)
multiple_means.show()

## 6. Relier la convergence au cas A/B

Nous suivons maintenant `avg(B) - avg(A)` avec les paramètres du
scénario de comparaison : `mu(A) = mu(B) = 103`, `sigma(A) = 18`,
`sigma(B) = 24` et 60 % de A. Chaque seed produit une nouvelle
trajectoire ; la valeur théorique de l'écart reste 0.

À `n = 2 000`, chaque échantillon contient 1 200 observations A et
800 observations B, comme dans l'exemple du projet.

In [ ]:
reference_mean = 103.0
reference_std_a = 18.0
reference_std_b = 24.0
reference_weight_a = 0.6
difference_seeds = list(range(1404, 1412))
difference_colors = [
    "#002D62", "#D94801", "#007F73", "#7B2CBF",
    "#8A6500", "#B51F4C", "#2F6B2F", "#4B5563",
]


def mean_difference_summary(seed):
    # Deux suites emboîtées, avec la même moyenne théorique dans A et B.
    maximum_size_a = int(sample_sizes[-1] * reference_weight_a)
    maximum_size_b = sample_sizes[-1] - maximum_size_a
    rng = np.random.default_rng(seed)
    values_a = rng.normal(reference_mean, reference_std_a, maximum_size_a)
    values_b = rng.normal(reference_mean, reference_std_b, maximum_size_b)
    sizes_a = (sample_sizes * reference_weight_a).astype(int)
    sizes_b = sample_sizes - sizes_a
    differences = [
        values_b[:size_b].mean() - values_a[:size_a].mean()
        for size_a, size_b in zip(sizes_a, sizes_b, strict=True)
    ]
    return pd.DataFrame(
        {
            "seed": seed,
            "sample_size": sample_sizes,
            "sample_size_a": sizes_a,
            "sample_size_b": sizes_b,
            "mean_difference": differences,
        }
    )


difference_summaries = {
    seed: mean_difference_summary(seed) for seed in difference_seeds
}
mean_differences = go.Figure()
for seed, color in zip(difference_seeds, difference_colors, strict=True):
    summary = difference_summaries[seed]
    mean_differences.add_scatter(
        x=summary["sample_size"],
        y=summary["mean_difference"],
        mode="lines+markers",
        name=f"seed = {seed}",
        line={"color": color, "width": 2.5},
    )
mean_differences.add_scatter(
    x=[sample_sizes.min(), sample_sizes.max()],
    y=[0, 0],
    mode="lines",
    name="mu(B) - mu(A) = 0",
    line={"color": "#202A33", "width": 3, "dash": "dash"},
)
mean_differences.add_vline(
    x=2_000,
    line={"color": "#6F8798", "width": 2, "dash": "dot"},
)
mean_differences.add_annotation(
    x=2_000,
    y=20,
    text="jeu observé<br>n = 2 000",
    showarrow=False,
    xanchor="left",
    yanchor="top",
)
style_figure(
    mean_differences,
    "seeds 1404 à 1411 · 20 valeurs de n par trajectoire",
    "avg(B) - avg(A) (€)",
)
mean_differences.show()

## Conclusion et limite

À `n = 5 000`, la trajectoire obtenue avec `seed = 406` donne une
moyenne proche de 100 et un écart-type proche de 15, sans leur être
exactement égale.

Ici, `mu` et `sigma` sont connus parce qu'ils ont servi à simuler
les données. Dans une étude réelle, ils sont généralement inconnus :
les statistiques de l'échantillon servent à les estimer.

Plusieurs trajectoires illustrent la stabilisation mais ne la
prouvent pas. Le `seed` modifie le chemin observé, pas les valeurs
théoriques du modèle.

La même logique s'applique à `avg(B) - avg(A)` : quand les deux
populations ont la même moyenne, l'écart observé fluctue autour de
0 et devient généralement plus stable quand `n` augmente.

Augmenter `n` réduit généralement la fluctuation d'échantillonnage ;
cela ne corrige ni un biais de sélection ni une erreur de mesure.